In [ ]:
!pip install transformers pillow

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

vqa = pipeline(
    task="visual-question-answering",
    model="dandelin/vilt-b32-finetuned-vqa"
)

"Кот"
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cat.png"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")


question = "What animal is in the picture?"
outputs = vqa(image=image, question=question)

print(outputs)

In [ ]:
"Другой вопрос"
question = "Is the cat sitting or standing?"
print(vqa(image=image, question=question))

### Вариант без пайплайна

In [ ]:
import torch
from transformers import ViltProcessor, ViltForQuestionAnswering
from PIL import Image
import requests
from io import BytesIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "dandelin/vilt-b32-finetuned-vqa"

processor = ViltProcessor.from_pretrained(model_name)
model = ViltForQuestionAnswering.from_pretrained(model_name).to(device)

# Картинка
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cat.png"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

question = "What animal is in the picture?"

# Подготовка входа
inputs = processor(
    images=image,
    text=question,
    return_tensors="pt"
).to(device)

# Инференс
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # [batch_size, num_labels]

# Индекс максимального логита
pred_id = logits.argmax(-1).item()

# Лейбл
answer = model.config.id2label[pred_id]
print("Answer:", answer)